This is for Task Human Validation

Step 1 — Configuration / file paths

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

RUBRIC_VERSION = "v6"
VALIDATION_N = 120

LABELS = [
    "Limited GenAI applicability",
    "Augmentation-oriented",
    "Automation-oriented",
    "Indeterminate mechanism",
]

BASE_DIR = Path(
    "/content/drive/MyDrive/GenAI-task-affordance-labor-outcomes-full/"
)

Human_Coder1 = (
    BASE_DIR
    / "02_Task_Data_and_Classification"
    / "Output"
    / "Coder1_Validation_v6.xlsx"
)

Human_Coder2 = (
    BASE_DIR
    / "02_Task_Data_and_Classification"
    / "Output"
    / "Coder2_Validation_v6.xlsx"
)

Human_Coder = (
    BASE_DIR
    / "02_Task_Data_and_Classification"
    / "Output"
    / "Coder3_Validation_v6.xlsx"
)


provisional_file = (
    BASE_DIR
    / "02_Task_Data_and_Classification"
    / "Output"
    / "Occupational_Task_Classification_Provisional_v6.csv"
)

provisional_task_classifications = pd.read_csv(provisional_file)

print(f"Provisional task classifications: {len(provisional_task_classifications):,}")

Provisional task classifications: 16,486


In [ ]:
coder1 = pd.read_excel(Human_Coder1)
coder2 = pd.read_excel(Human_Coder2)

print(coder1.shape, coder2.shape)
print(coder1.columns.tolist())
print(coder2.columns.tolist())

(120, 6) (120, 6)
['task_id', 'soc6', 'occupation_title', 'task_statement', 'coder_label', 'coder_notes']
['task_id', 'soc6', 'occupation_title', 'task_statement', 'coder_label', 'coder_notes']


In [ ]:
validation = (
    coder1
    .merge(
        coder2,
        on="task_id",
        suffixes=("_c1", "_c2"),
        validate="one_to_one"
    )
    .merge(
        provisional_task_classifications[
            ["task_id", "task_statement", "provisional_label"]
        ],
        on="task_id",
        how="left",
        validate="one_to_one"
    )
)
print("Matched tasks:", len(validation))
print("Missing provisional labels:", validation["provisional_label"].isna().sum())
print(
    "Coder statements identical:",
    validation["task_statement_c1"].eq(
        validation["task_statement_c2"]
    ).sum()
)
print(
    "Current statements unchanged:",
    validation["task_statement_c1"].eq(
        validation["task_statement"]
    ).sum()
)

print("\nCoder 1 labels:")
print(coder1["coder_label"].value_counts(dropna=False))

print("\nCoder 2 labels:")
print(coder2["coder_label"].value_counts(dropna=False))

Matched tasks: 120
Missing provisional labels: 42
Coder statements identical: 120
Current statements unchanged: 78

Coder 1 labels:
coder_label
Augmentation-oriented          62
Automation-oriented            28
Limited GenAI applicability    28
Indeterminate mechanism         2
Name: count, dtype: int64

Coder 2 labels:
coder_label
Augmentation-oriented          66
Limited GenAI applicability    26
Automation-oriented            23
Indeterminate mechanism         5
Name: count, dtype: int64


In [ ]:
human_tasks = coder1[
    ["task_id", "soc6", "task_statement"]
].copy()

current_tasks = provisional_task_classifications[
    ["task_id", "soc6", "task_statement", "provisional_label"]
].copy()

for df in [human_tasks, current_tasks]:
    df["match_key"] = (
        df["soc6"].astype(str).str.strip()
        + "|"
        + df["task_statement"]
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
    )
missing_tasks = human_tasks[
    ~human_tasks["task_id"].isin(current_tasks["task_id"])
]
statement_matches = missing_tasks.merge(
    current_tasks[
        ["task_id", "match_key", "provisional_label"]
    ],
    on="match_key",
    how="left",
    suffixes=("_old", "_current")
)
print("Missing by task ID:", len(missing_tasks))
print(
    "Recovered by SOC + task statement:",
    statement_matches["provisional_label"].notna().sum()
)
print(
    "Still unmatched:",
    statement_matches["provisional_label"].isna().sum()
)

Missing by task ID: 42
Recovered by SOC + task statement: 0
Still unmatched: 42


In [ ]:
current_validation = validation[
    validation["provisional_label"].notna()
].copy()

print("Usable validation tasks:", len(current_validation))
print("SOC6 occupations represented:", current_validation["soc6_c1"].nunique())

for column in [
    "coder_label_c1",
    "coder_label_c2",
    "provisional_label"
]:
    print(f"\n{column}:")
    print(current_validation[column].value_counts())

Usable validation tasks: 78
SOC6 occupations represented: 36

coder_label_c1:
coder_label_c1
Augmentation-oriented          38
Automation-oriented            21
Limited GenAI applicability    17
Indeterminate mechanism         2
Name: count, dtype: int64

coder_label_c2:
coder_label_c2
Augmentation-oriented          36
Automation-oriented            20
Limited GenAI applicability    17
Indeterminate mechanism         5
Name: count, dtype: int64

provisional_label:
provisional_label
Augmentation-oriented          42
Limited GenAI applicability    24
Automation-oriented            11
Indeterminate mechanism         1
Name: count, dtype: int64


In [ ]:
human_agreement = (
    current_validation["coder_label_c1"]
    == current_validation["coder_label_c2"]
)

print("Human–human agreements:", human_agreement.sum())
print("Human–human disagreements:", (~human_agreement).sum())
print("Agreement rate:", f"{human_agreement.mean():.1%}")

human_kappa = cohen_kappa_score(
    current_validation["coder_label_c1"],
    current_validation["coder_label_c2"]
)

print("Cohen's kappa:", f"{human_kappa:.3f}")

pd.crosstab(
    current_validation["coder_label_c1"],
    current_validation["coder_label_c2"]
)

Human–human agreements: 63
Human–human disagreements: 15
Agreement rate: 80.8%
Cohen's kappa: 0.707


coder_label_c2,Augmentation-oriented,Automation-oriented,Indeterminate mechanism,Limited GenAI applicability
coder_label_c1,,,,
Augmentation-oriented,30,0,2,6
Automation-oriented,1,20,0,0
Indeterminate mechanism,0,0,2,0
Limited GenAI applicability,5,0,1,11


In [ ]:
model_human_results = []

for name, column in {
    "Model vs Coder 1": "coder_label_c1",
    "Model vs Coder 2": "coder_label_c2"
}.items():

    agreement = (
        current_validation["provisional_label"]
        == current_validation[column]
    )

    model_human_results.append({
        "comparison": name,
        "n": len(current_validation),
        "agreements": agreement.sum(),
        "agreement_rate": agreement.mean(),
        "cohen_kappa": cohen_kappa_score(
            current_validation["provisional_label"],
            current_validation[column]
        )
    })

model_human_results = pd.DataFrame(model_human_results)
display(model_human_results)

,comparison,n,agreements,agreement_rate,cohen_kappa
0,Model vs Coder 1,78,56,0.717949,0.553938
1,Model vs Coder 2,78,53,0.679487,0.504950


In [ ]:
for name, column in {
    "Model vs Coder 1": "coder_label_c1",
    "Model vs Coder 2": "coder_label_c2"
}.items():

    print(f"\n{name}")

    matrix = pd.crosstab(
        current_validation["provisional_label"],
        current_validation[column]
    ).reindex(
        index=LABELS,
        columns=LABELS,
        fill_value=0
    )

    display(matrix)


Model vs Coder 1


coder_label_c1,Limited GenAI applicability,Augmentation-oriented,Automation-oriented,Indeterminate mechanism
provisional_label,,,,
Limited GenAI applicability,15,7,1,1
Augmentation-oriented,2,30,9,1
Automation-oriented,0,0,11,0
Indeterminate mechanism,0,1,0,0



Model vs Coder 2


coder_label_c2,Limited GenAI applicability,Augmentation-oriented,Automation-oriented,Indeterminate mechanism
provisional_label,,,,
Limited GenAI applicability,14,7,1,2
Augmentation-oriented,3,28,8,3
Automation-oriented,0,0,11,0
Indeterminate mechanism,0,1,0,0


In [ ]:
for name, column in {
    "Model–Coder 1": "coder_label_c1",
    "Model–Coder 2": "coder_label_c2",
}.items():
    matrix = pd.crosstab(
        current_validation["provisional_label"],
        current_validation[column],
        dropna=False,
    )
    displayed = matrix.reindex(
        index=LABELS, columns=LABELS, fill_value=0
    )

    print(name)
    print("Validation records:", len(current_validation))
    print("Complete matrix total:", matrix.to_numpy().sum())
    print("Displayed matrix total:", displayed.to_numpy().sum())

Model–Coder 1
Validation records: 78
Complete matrix total: 78
Displayed matrix total: 78
Model–Coder 2
Validation records: 78
Complete matrix total: 78
Displayed matrix total: 78
